In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("../datasets/customer_support_tickets.csv")
df.head()
df.info()
df.columns
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   str    
 2   Customer Email                8469 non-null   str    
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   str    
 5   Product Purchased             8469 non-null   str    
 6   Date of Purchase              8469 non-null   str    
 7   Ticket Type                   8469 non-null   str    
 8   Ticket Subject                8469 non-null   str    
 9   Ticket Description            8469 non-null   str    
 10  Ticket Status                 8469 non-null   str    
 11  Resolution                    2769 non-null   str    
 12  Ticket Priority               8469 non-null   str    
 13  Ticket Channel

(8469, 17)

In [5]:
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [6]:
df.isnull().sum()

Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                      5700
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64

#renaming columns


In [7]:
new_df = pd.DataFrame()

new_df["complaint_id"] = df["Ticket ID"]

new_df["complaint_text"] = (
    df["Ticket Subject"].fillna("") + ". " +
    df["Ticket Description"].fillna("")
)

new_df["category"] = df["Ticket Type"]

new_df["priority"] = df["Ticket Priority"]

generating sentiments column


In [8]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

texts = new_df["complaint_text"].fillna("").tolist()

sentiment_results = sentiment_model(
    texts,
    batch_size=32,
    truncation=True
)

new_df["sentiment"] = [
    result["label"].capitalize()
    for result in sentiment_results
]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generating emotions column

In [ ]:
emotion_model = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=1
)
emotion_results = emotion_model(
    texts,
    batch_size=32,
    truncation=True
)

new_df["raw_emotion"] = [
    result[0]["label"]
    for result in emotion_results
]

mapping emotions

In [ ]:
def map_emotion(emotion):

    if emotion == "anger":
        return "Angry"

    elif emotion in ["annoyance", "disapproval"]:
        return "Frustrated"

    elif emotion in [
        "sadness",
        "disappointment",
        "grief",
        "remorse"
    ]:
        return "Sad"

    elif emotion in [
        "joy",
        "gratitude",
        "love",
        "relief",
        "optimism"
    ]:
        return "Satisfied"

    else:
        return "Not Satisfied"



In [ ]:
raw_emotions = [
    result[0]["label"]
    for result in emotion_results
]

new_df["emotion"] = [
    map_emotion(emotion)
    for emotion in raw_emotions
]

In [ ]:
# Select only the required columns

final_df = new_df[[
    "complaint_id",
    "complaint_text",
    "category",
    "priority",
    "sentiment",
    "emotion",
    "department"
]].copy()

# Check the dataset
print(final_df.head())

# Check unique labels
print("Sentiments:", final_df["sentiment"].unique())
print("Emotions:", final_df["emotion"].unique())

# Check missing values
print(final_df.isnull().sum())

# Export dataset
final_df.to_csv(
    "final_dataset.csv",
    index=False
)